# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id and fields
print('Available record sets:')
for rs in meta.record_sets:
    print(f"  RecordSet @id: {rs.id}  |  Name: {getattr(rs, 'name', rs.id)}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - Field @id: {field.id}" + (f" | Name: {getattr(field, 'name', field.id)}" if hasattr(field, 'name') else ""))
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into Pandas DataFrames

# ---- Customize here if your dataset contains multiple record sets ----
# For example, assume the main data is in a record set with a known @id.
# We'll discover available record sets and pick the first for demonstration purposes.
record_sets_ids = [rs.id for rs in meta.record_sets]
print(f"Found record sets: {record_sets_ids}")
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nLoaded {df.shape[0]} records from record set '{record_set_id}'. Sample columns:")
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select record set for EDA
main_record_set_id = record_sets_ids[0]  # Choose adjust according to what was printed
df = dataframes[main_record_set_id]

print(f"Fields in record set '{main_record_set_id}':")
print(df.columns.tolist())

# Assume a likely numeric field (e.g., 'MW_kDa', 'Abundance', etc.); update accordingly.
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Pick the first numeric field for demonstration
else:
    print('No numeric field found for EDA.')
    numeric_field_id = None

if numeric_field_id:
    # Remove records with missing values to avoid errors
    df_clean = df[df[numeric_field_id].notnull()]
    threshold = df_clean[numeric_field_id].mean()  # Filter values above mean
    filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a non-numeric field
    group_candidates = [col for col in df.columns if (col != numeric_field_id and (df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col])))]
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head(4))
    else:
        print('No suitable grouping field found for groupby example.')
else:
    print('No numeric field available for EDA in this record set.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals():
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.